# Prompt-only Baseline

Use this notebook to run the no-post-training baseline on `phase1_test.jsonl`.

Recommended default:

- model: `Qwen/Qwen2.5-3B-Instruct`
- input: `data/samples/phase1_test.jsonl`
- output: `results/predictions/qwen25_3b_prompt_only_test.jsonl`

In [ ]:
# Run this once in a fresh Jupyter environment if dependencies are missing.

%pip install -q transformers datasets accelerate pyyaml jsonschema

In [ ]:
from pathlib import Path
import os


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root from current working directory.')


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
import torch
import platform
import sys

print('python =', sys.version)
print('platform =', platform.platform())
print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    print('gpu_count =', torch.cuda.device_count())
    print('bf16_supported =', torch.cuda.is_bf16_supported())

In [ ]:
from pathlib import Path
import json

base_model = 'Qwen/Qwen2.5-3B-Instruct'
test_path = PROJECT_ROOT / 'data' / 'samples' / 'phase1_test.jsonl'
prediction_dir = PROJECT_ROOT / 'results' / 'predictions'
prediction_dir.mkdir(parents=True, exist_ok=True)
prediction_path = prediction_dir / 'qwen25_3b_prompt_only_test.jsonl'

print('test_exists =', test_path.exists(), test_path)
print('prediction_path =', prediction_path)

test_records = []
with test_path.open('r', encoding='utf-8') as handle:
    for line in handle:
        test_records.append(json.loads(line))

len(test_records), test_records[0]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else torch.float16

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=dtype,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
print('model loaded')

In [ ]:
generation_kwargs = {
    'max_new_tokens': 256,
    'do_sample': False,
    'temperature': 1.0,
    'top_p': 1.0,
}

generation_kwargs

In [ ]:
def build_inference_messages(record):
    return [
        {
            'role': 'system',
            'content': 'You are an information extraction model. Return only JSON that matches the requested schema. Do not add explanations or markdown.',
        },
        {
            'role': 'user',
            'content': (
                f"Task: extract a structured record for {record['task_name']}.\n"
                f"Schema name: {record['schema_name']}\n"
                'Return a JSON object only.\n'
                'Input text:\n'
                f"{record['input_text']}"
            ),
        },
    ]


def generate_prediction_text(record):
    messages = build_inference_messages(record)
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, **generation_kwargs)
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


sample_pred = generate_prediction_text(test_records[0])
print(sample_pred)

In [ ]:
predictions = []

for idx, record in enumerate(test_records):
    prediction_text = generate_prediction_text(record)
    prediction_json = None
    try:
        prediction_json = json.loads(prediction_text)
    except json.JSONDecodeError:
        prediction_json = None

    predictions.append(
        {
            'sample_id': record['sample_id'],
            'prediction_text': prediction_text,
            'prediction_json': prediction_json,
            'metadata': {
                'model_name': base_model,
                'experiment_id': 'qwen25_3b_prompt_only_v1',
            },
        }
    )

    if (idx + 1) % 25 == 0:
        print(f'generated {idx + 1} / {len(test_records)}')

len(predictions)

In [ ]:
with prediction_path.open('w', encoding='utf-8') as handle:
    for record in predictions:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')

print('saved predictions to', prediction_path)
print('exists =', prediction_path.exists())
print('size =', prediction_path.stat().st_size if prediction_path.exists() else None)

In [ ]:
with prediction_path.open('r', encoding='utf-8') as handle:
    for i, line in enumerate(handle):
        if i >= 2:
            break
        print(json.loads(line))
        print('---')